# Loading and Preprocessing the downloaded data

## Import Block
Import all packages you need throughout this script

In [35]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

# Function Definitions

In [80]:
def check_and_convert_crs(gdf, epsg=2056):
    '''
    This function is used to check whether the loaded data is in the correct crs.
    If the data is already in the right projection, the function does nothing.
    If no crs is assigned yet the function throws and error

    Parameters:
    gdf: geodataframe
            The geodataframe we want to check for correct crs.
    epsg: int,
            The crs we want our data to be in. Default is 2056 (CH1903+ / LV95). 
    
    Returns:
    gdf: geodataframe
            The same geodataframe we put into the function. Unchanged if the crs was correct, reprojected if the crs was not correct.
    '''
    #Block one to check if the data has a crs
    if gdf.crs.to_epsg() is None:
        print("No CRS defined!")
        return gdf
    
    #Block two to check for the correct crs
    if gdf.crs.to_epsg() != epsg:
        gdf=gdf.to_crs(epsg=epsg)
        return gdf
    
    return gdf

def check_active_geometry_name(gdf):
    '''
    This function has the sole purpose to give the name of the active geometry of a geodataframe or to give an error message if none is defined.

    Parameters:
    gdf: geodataframe
        The geodataframe you want to check.

    Retruns:
    no return values    
    '''
    if gdf.geometry is None:
        print("NO GEMOETRY DEFINED! PLEASE ASSIGN THE CORRECT COLUMN BEFORE PROCEEDING!")

    else:
        print(f"The active geometry column of this geodataframe is called '{gdf.geometry.name}'.")
    return

# Neighborhoods
The neighborhood data consists of 3 layers. In the metadata the layers are introduced as follows (in german): 

### 1 ADM_STATISTISCHE_QUARTIERE_B_P
Beschriftungspunkte der statistischen Quartiere. Dieser Punktlayer beinhaltet Informationen zur
optimalen Positionierung von Beschriftungen in Karten.

### 2 ADM_STATISTISCHE_QUARTIERE_MAP
Dieser Layer kann für Datenvisualisierungen und kartographische Darstellungen verwendet werden.
Er enthält die vollständige Attributtabelle und ist generalisiert, sprich: die Geometrie wurde
vereinfacht und entspricht nicht der - aus Sicht der amtlichen Vermessung - korrekten Lage.

### 3 ADM_STATISTISCHE_QUARTIERE_V
Dieser Layer enthält die exakten Grenzverläufe der Statistischen Quartiere und enthält die
vollständige Attributtabelle. Dieser Layer eignet sich zur Berechnung von Flächenangaben oder bei räumlichen Verschnitten. Als kartographischer Layer ist er jedoch nicht geeignet.

The full information, and the attribute tables associated with each layer, can be found in the data/raw/ folder. The file is named "metadaten_quartiere.pdf".

From this information, I selected the third layer for the spatial join with the Züriwieneu reports. For the subsequent visualisations, the other two layers will be usefull, therefore they are prepared as well.



### Loading Data
First the data is loaded and checked for the correct crs.

In [36]:
#defining the path of this file:
quartiere_raw_path=Path("../data/raw/quartiere.gpkg")


#loading the layer for subsequent spatial join:
quartiere_processing=gpd.read_file(quartiere_raw_path, layer="stzh.adm_statistische_quartiere_v")
quartiere_processing=check_and_convert_crs(quartiere_processing)

#loading the layer for subsequent cartographic visualisations: 
quartiere_map=gpd.read_file(quartiere_raw_path, layer="stzh.adm_statistische_quartiere_map")
quartiere_map=check_and_convert_crs(quartiere_map)

#loading the layer for subsequent labeling in cartogrpahic visualisations:
quartiere_labels=gpd.read_file(quartiere_raw_path, layer="stzh.adm_statistische_quartiere_b_p")
quartiere_labels=check_and_convert_crs(quartiere_labels)

## 1. Quartiere_labels
This layer will only be used to add the labels to a cartographic map. Therfore, most attributes in the attribute table are not needed. The attribute table is given below: 

| Name      | Typ          | Beschreibung                                                                  |
|------------|---------------|--------------------------------------------------------------------------------|
| ORI        | DOUBLE        | Rotation der Beschriftung; undefiniert = 100.0                                |
| HALI       | LONG INTEGER  | Horizontaler Versatz zum Ursprungspunkt; undefiniert = Center                 |
| OBJECTID   | LONG INTEGER  | ESRI System-Identifikator. Nicht stabil über Zeit.                            |
| VALI       | LONG INTEGER  | Vertikaler Versatz zum Ursprungspunkt; undefiniert = Half                     |
| GEOMETRIE  | SHAPE         | Geometriefeld                                                                 |
| KUERZEL    | STRING        | -                                                                              |
| OBJID      | STRING        | System-Identifikator. Nicht stabil über Zeit.                                 |
| NAME       | STRING        | Bezeichnung des statistischen Quartiers                                       |


Only the name, kuerzel and geometry is kept. ORI, HALI and VALI could be usefull as well, but as all attributes have the same values (ORI=0.0, HALI=1.0 and VALI=2.0), they can be dropped as well. 

After dropping all unnecessary columns, we want to check wheter an active geometry column exists. 

In [47]:
#check if the column names are identical to the attribute table in the metadata and verify the active geometry and the number of NaN visually
quartiere_labels.info()

# only keep the columns you need
quartiere_labels=quartiere_labels[["name", "kuerzel", "geometry"]]

#check for the name of the active geometry column
check_active_geometry_name(quartiere_labels)

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   name      34 non-null     str     
 1   kuerzel   34 non-null     str     
 2   geometry  34 non-null     geometry
dtypes: geometry(1), str(2)
memory usage: 1.3 KB
The active geometry column of this geodataframe is called 'geometry'.


As an active geometry exists, we export the processed file into the data/processed/ folder. 

In [52]:
processed_quartiere_path=Path("../data/processed/quartiere_labels.gpkg")
quartiere_labels.to_file(processed_quartiere_path, driver="GPKG")

## 2. Quartiere_map
This layer will only be used as the neighborhood-geometry for visualisations. The attribute table is given below:

| Name      | Typ          | Beschreibung                                                   |
|------------|---------------|-----------------------------------------------------------------|
| KNR        | DOUBLE        | Offizielle ID des Stadtkreises.                                |
| QNR        | DOUBLE        | Offizielle ID des Statistischen Quartiers.                     |
| OBJECTID   | LONG INTEGER  | Interner System-Identifikator. Nicht stabil über Zeit.         |
| GEOMETRIE  | SHAPE         | Geometriefeld.                                                 |
| KNAME      | STRING        | Name des Stadtkreises.                                         |
| OBJID      | STRING        | System-Identifikator. Nicht stabil über Zeit.                  |
| QNAME      | STRING        | Name des Statistischen Quartiers.                              |

From this attribute table, we can drop the OBJECTID and OBJID as they are not stable over time. The rest is not bad to keep.  
Additionally we need to make sure that the GEOMETRIE column is actually a geometry data type.


In [51]:
# verify the column names. For example, the objid column does not exist! 
quartiere_map.info()

#so we have to get rid of the column objectid.
quartiere_map=quartiere_map[["qnr", "qname", "knr", "kname", "geometry"]]

check_active_geometry_name(quartiere_map)

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   qnr       34 non-null     int32   
 1   qname     34 non-null     str     
 2   knr       34 non-null     int32   
 3   kname     34 non-null     str     
 4   geometry  34 non-null     geometry
dtypes: geometry(1), int32(2), str(2)
memory usage: 1.7 KB
The active geometry column of this geodataframe is called 'geometry'.


In [53]:
processed_quartiere_path=Path("../data/processed/quartiere_map.gpkg")
quartiere_labels.to_file(processed_quartiere_path, driver="GPKG")

## 3. Quartiere_processing
This is the layer that will be used for the spatial join with the reports of the ZüriWieNeu-Website. The attribute table looks as follows: 

| Name      | Typ          | Beschreibung                                                   |
|------------|---------------|-----------------------------------------------------------------|
| QNR        | DOUBLE        | Offizielle ID des Statistischen Quartiers.                     |
| KNR        | DOUBLE        | Offizielle ID des Stadtkreises.                                |
| OBJECTID   | LONG INTEGER  | Interner System-Identifikator. Nicht stabil über Zeit.         |
| GEOMETRIE  | SHAPE         | Geometriefeld.                                                 |
| KNAME      | STRING        | Name des Stadtkreises.                                         |
| OBJID      | STRING        | System-Identifikator. Nicht stabil über Zeit.                  |
| QNAME      | STRING        | Name des Statistischen Quartiers.                              |

So, the OBJID and OBJECTID are redundant and will be excluded for the future analysis

In [56]:
# verify the column names. For example, the objid column does not exist! 
quartiere_processing.info()

#so we have to get rid of the column objectid.
quartiere_processing=quartiere_processing[["qnr", "qname", "knr", "kname", "geometry"]]

check_active_geometry_name(quartiere_processing)

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   objid     34 non-null     str     
 1   objectid  34 non-null     int32   
 2   qname     34 non-null     str     
 3   qnr       34 non-null     int32   
 4   kname     34 non-null     str     
 5   knr       34 non-null     int32   
 6   geometry  34 non-null     geometry
dtypes: geometry(1), int32(3), str(3)
memory usage: 2.2 KB
The active geometry column of this geodataframe is called 'geometry'.


For question 1, it would be nice to compare the absolute number of reports normalized by the area of each neighborhood! Therefore, I add the area of each neighborhood to the gdf.

In [74]:
# creating a column containing each neighborhoods area in km^2
quartiere_processing["area_km2"]=(quartiere_processing.area/1000000)#the /1'000'000 is needed to get the area in km^2

# check wheter the result is approximately correct:
quartiere_processing["area_km2"].sum() #according to https://www.stadt-zuerich.ch/de/bildung/volksschule/unterrichtsmaterial/gang-dur-zueri/zuerich-in-zahlen.html, the area of the city of Zurich is 91.9 km^2



np.float64(91.87980054046378)

And now finally exporting the gdf.

In [92]:
# one last sneak peak to verify everything worked:
display(quartiere_processing.head(5))

# define the path and export it
processed_quartiere_path=Path("../data/processed/quartiere_processed.gpkg")
quartiere_processing.to_file(processed_quartiere_path, driver="GPKG")

,qnr,qname,knr,kname,geometry,area_km2
0,31,Alt-Wiedikon,3,Kreis 3,"POLYGON ((2680606.662 1247034.584, 2680626.356...",1.692468
1,74,Witikon,7,Kreis 7,"POLYGON ((2685858.632 1246502.629, 2685860.738...",4.933788
2,42,Langstrasse,4,Kreis 4,"POLYGON ((2681313.304 1248613.857, 2681459.605...",1.211798
3,52,Escher Wyss,5,Kreis 5,"POLYGON ((2680009.144 1249565.021, 2680055.843...",1.266279
4,24,Enge,2,Kreis 2,"POLYGON ((2681898.171 1246379.668, 2681899.115...",2.365206


### Note on the workflows so far:
Until now, the number of non-null object within each column was identical to the total number of rows. Therefore, no NaN-values are present in the data and we do not have to account for it here.

# Reports of the Züri Wie Neu platform
This dataset contains all reports made by users online. The attribute table is given below:

| Name                  | Typ          | Beschreibung |
|-----------------------|---------------|---------------|
| AGENCY_SENT_DATETIME  | DATE          | Zeitpunkt der Benachrichtigung der verantwortlichen Stelle in der Stadtverwaltung [DD.MM.YYYY HH:MM:SS]. |
| REQUESTED_DATETIME    | DATE          | Datum und Zeitpunkt, an welchem die Meldung erstellt wurde [DD.MM.YYYY HH:MM:SS]. |
| UPDATED_DATETIME      | DATE          | Datum und Zeitpunkt, an welchem die Meldung zuletzt verändert wurde bzw. abgeschlossen wurde [DD.MM.YYYY HH:MM:SS]. |
| E                     | DOUBLE        | Ostkoordinate, CH1903+ / LV95. |
| N                     | DOUBLE        | Nordkoordinate, CH1903+ / LV95. |
| OBJECTID              | LONG INTEGER  | System-Identifikator. Nicht stabil über Zeit. |
| GEOMETRIE             | SHAPE         | Geometriefeld. |
| URL                   | STRING        | URL der Meldung auf „Züri wie neu“. |
| INTERFACE_USED        | STRING        | Verwendetes Interface [Web Interface, iPad, iPhone, Android]. |
| MEDIA_URL             | STRING        | URL zu den von der meldenden Person übermittelten Fotos. |
| DETAIL                | STRING        | Meldungsbeschreibung. |
| SERVICE_CODE          | STRING        | Kategorie [Abfall/Sammelstelle, Beleuchtung/Uhren, Graffiti, Signalisation/Lichtsignal, Grünflächen/Spielplätze, Strasse/Trottoir/Platz, Brunnen/Hydranten, VBZ/ÖV]. |
| SERVICE_NOTICE        | STRING        | Antwort der Stadtverwaltung. |
| SERVICE_REQUEST_ID    | STRING        | Eindeutige „Züri wie neu“-System-ID der Meldung. |
| STATUS                | STRING        | Bearbeitungsstatus [erfasst, aufgenommen, in Bearbeitung, beantwortet]. |
| TITLE                 | STRING        | Titel der Meldung. |
| DESCRIPTION           | STRING        | Kombination aus Titel und Meldungsbeschreibung. |
| SERVICE_NAME          | STRING        | Kategorie [Abfall/Sammelstelle, Beleuchtung/Uhren, Graffiti, Signalisation/Lichtsignal, Grünflächen/Spielplätze, Strasse/Trottoir/Platz, Brunnen/Hydranten, VBZ/ÖV]. |

Here, a lot of attributes we are not too interested in are present. I will select the following attributes for further analysis: REQUESTED_DATETIME, E, N, GEOMETRIE, SERVICE_CODE, SERVICE REQUEST_ID, DESCRIPTION.

In [67]:
#first load the dataset:
raw_requests_path=Path("../data/raw/reports_ZHwieneu.gpkg")
reports_gdf=gpd.read_file(raw_requests_path)

#check the crs:
reports_gdf=check_and_convert_crs(reports_gdf)

#check for the column names:
reports_gdf.info() #also here no NaNs!!!

#selecting only the columns needed:
reports_gdf=reports_gdf[["service_request_id", "requested_datetime", "e", "n", "geometry", "service_code", "description"]]

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 73243 entries, 0 to 73242
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   objectid              73243 non-null  int32         
 1   service_request_id    73243 non-null  str           
 2   requested_datetime    73243 non-null  datetime64[ms]
 3   agency_sent_datetime  72411 non-null  datetime64[ms]
 4   updated_datetime      73243 non-null  datetime64[ms]
 5   e                     73243 non-null  int32         
 6   n                     73243 non-null  int32         
 7   service_code          73243 non-null  str           
 8   service_name          73243 non-null  str           
 9   status                73243 non-null  str           
 10  userid                73243 non-null  int32         
 11  title                 73243 non-null  str           
 12  detail                73243 non-null  str           
 13  media_ur

1. Check wheter it worked.
2. As we now have a unambigous id for each report (the service_request_id column), I will see if I can set it as the new index.  

In [68]:
#check if the selection worked
display(reports_gdf.head(10))

#checking wheter every service_request_id is really unique
reports_gdf["service_request_id"].nunique() #--> it is, but as some numbers are missing, I will keep the automatic index (could be important in a for loop eventually) 


,service_request_id,requested_datetime,e,n,geometry,service_code,description
0,1,2013-03-14 15:16:15,2678968,1247548,POINT (2678968 1247548),Strasse/Trottoir/Platz,Auf dem Asp: Auf dem Asphalt des Bürgersteigs ...
1,2,2013-03-14 15:17:57,2680746,1249916,POINT (2680746 1249916),Strasse/Trottoir/Platz,Vermessungs: Vermessungspunkt ist nicht mehr b...
2,4,2013-03-15 09:14:16,2684605,1251431,POINT (2684605 1251431),Strasse/Trottoir/Platz,Beim Trotto: Beim Trottoir sind einige Randste...
3,5,2013-03-15 09:17:15,2681754,1250376,POINT (2681754 1250376),Strasse/Trottoir/Platz,Auf dem Par: Auf dem Parkplatz beim Waidspital...
4,6,2013-03-15 10:36:53,2683094,1247762,POINT (2683094 1247762),Abfall/Sammelstelle,Arbeitskist: Arbeitskiste ist rund herum versc...
5,7,2013-03-16 17:54:42,2683475,1247422,POINT (2683475 1247422),Strasse/Trottoir/Platz,Gelockerte : Gelockerte und fehlende Pflasters...
6,8,2013-03-16 18:04:21,2683303,1247675,POINT (2683303 1247675),Strasse/Trottoir/Platz,Steinerne B: Steinerne Ballustrade bei der Ura...
7,9,2013-03-18 07:06:53,2678769,1248921,POINT (2678769 1248921),Strasse/Trottoir/Platz,"Feldblumens: Feldblumenstrasse 30, Schlagloch"
8,10,2013-03-18 07:16:54,2678479,1248732,POINT (2678479 1248732),Strasse/Trottoir/Platz,Rautistrass: Rautistrasse / Friedhofstrasse: S...
9,11,2013-03-20 07:20:37,2682529,1251346,POINT (2682529 1251346),Strasse/Trottoir/Platz,Neben dem S: Neben dem Schacht sind ca. 2 m Wa...


73243

finally export the processed report-data

In [78]:
processed_report_path=Path("../data/processed/reports_processed.gpkg")
reports_gdf.to_file(processed_report_path, driver="GPKG")

# Spatial join --> DOES NOT WORK YET!!!

The last prerequisite for then subsequently investigating the 4 questions, is to combine the two datasets. For this a spatial join is used.

* We will use the spatial predicate "intersects" as we do not care wheter the report was made in the middle of a neighborhood or on the border. BUT, this method requires us to manually check wheter a point is only assigned to one neighborhood. 
* Further, I want to attach the neighborhodd info to the reports. Therfore the join type left (with reports as the left input). *Why not using inner?* Although inner would be easier, it yields the possibility that reports are automatically deleted. I preferably do that after myself. 

In [93]:
# reaload the processed files again
# first define the paths
qrt_path=Path("../data/processed/quartiere_processed.gpkg")
rpt_path=Path("../data/processed/reports_processed.gpkg")

#load quartiere
quartiere_processed=gpd.read_file(qrt_path)
quartiere_processed=check_and_convert_crs(quartiere_processed)

#load reports
reports_processed=gpd.read_file(rpt_path)
reports_processed=check_and_convert_crs(reports_processed)

quartiere_processed.head(5)


,qnr,qname,knr,kname,area_km2,geometry
0,31,Alt-Wiedikon,3,Kreis 3,1.692468,"POLYGON ((2680606.662 1247034.584, 2680626.356..."
1,74,Witikon,7,Kreis 7,4.933788,"POLYGON ((2685858.632 1246502.629, 2685860.738..."
2,42,Langstrasse,4,Kreis 4,1.211798,"POLYGON ((2681313.304 1248613.857, 2681459.605..."
3,52,Escher Wyss,5,Kreis 5,1.266279,"POLYGON ((2680009.144 1249565.021, 2680055.843..."
4,24,Enge,2,Kreis 2,2.365206,"POLYGON ((2681898.171 1246379.668, 2681899.115..."


In [96]:
reports_and_quartiere=gpd.sjoin(reports_processed, quartiere_processed, how="left", predicate="intersects")

reports_and_quartiere.head(20)

,service_request_id,requested_datetime,e,n,service_code,description,geometry,index_right,qnr,qname,knr,kname,area_km2
0,1,2013-03-14 15:16:15,2678968,1247548,Strasse/Trottoir/Platz,Auf dem Asp: Auf dem Asphalt des Bürgersteigs ...,POINT (2678968 1247548),16,91,Albisrieden,9,Kreis 9,4.614738
1,2,2013-03-14 15:17:57,2680746,1249916,Strasse/Trottoir/Platz,Vermessungs: Vermessungspunkt ist nicht mehr b...,POINT (2680746 1249916),20,101,Höngg,10,Kreis 10,6.983616
2,4,2013-03-15 09:14:16,2684605,1251431,Strasse/Trottoir/Platz,Beim Trotto: Beim Trottoir sind einige Randste...,POINT (2684605 1251431),33,121,Saatlen,12,Kreis 12,1.128136
3,5,2013-03-15 09:17:15,2681754,1250376,Strasse/Trottoir/Platz,Auf dem Par: Auf dem Parkplatz beim Waidspital...,POINT (2681754 1250376),21,102,Wipkingen,10,Kreis 10,2.102056
4,6,2013-03-15 10:36:53,2683094,1247762,Abfall/Sammelstelle,Arbeitskist: Arbeitskiste ist rund herum versc...,POINT (2683094 1247762),15,14,City,1,Kreis 1,0.595057
5,7,2013-03-16 17:54:42,2683475,1247422,Strasse/Trottoir/Platz,Gelockerte : Gelockerte und fehlende Pflasters...,POINT (2683475 1247422),13,11,Rathaus,1,Kreis 1,0.357099
6,8,2013-03-16 18:04:21,2683303,1247675,Strasse/Trottoir/Platz,Steinerne B: Steinerne Ballustrade bei der Ura...,POINT (2683303 1247675),24,13,Lindenhof,1,Kreis 1,0.266557
7,9,2013-03-18 07:06:53,2678769,1248921,Strasse/Trottoir/Platz,"Feldblumens: Feldblumenstrasse 30, Schlagloch",POINT (2678769 1248921),28,92,Altstetten,9,Kreis 9,7.442099
8,10,2013-03-18 07:16:54,2678479,1248732,Strasse/Trottoir/Platz,Rautistrass: Rautistrasse / Friedhofstrasse: S...,POINT (2678479 1248732),28,92,Altstetten,9,Kreis 9,7.442099
9,11,2013-03-20 07:20:37,2682529,1251346,Strasse/Trottoir/Platz,Neben dem S: Neben dem Schacht sind ca. 2 m Wa...,POINT (2682529 1251346),23,61,Unterstrass,6,Kreis 6,2.466186


In [ ]:
reports_and_quartiere["qnr"].isna().sum() #also no NaN --> all points were assigned to neighborhood 

np.int64(0)